In [1]:
import re
from bs4 import BeautifulSoup

def linearize_sec_table(table_soup):
    """
    Converte una tabella HTML SEC in una lista di frasi semantiche (Linearizzazione).
    """
    rows = table_soup.find_all("tr")
    if not rows:
        return ""

    # 1. ESTRAZIONE DEGLI HEADER
    # Le tabelle SEC spesso non usano i tag <th>, ma <td> standard.
    # Assumiamo che la prima riga contenga gli anni/intestazioni.
    headers = []
    header_row = rows[0]
    for col in header_row.find_all(["td", "th"]):
        text = col.get_text(strip=True)
        # CRITICO: Se la cella è vuota (es. padding), manteniamo un placeholder.
        # Questo impedisce lo slittamento delle colonne!
        headers.append(text if text else "BLANK_COL")

    linearized_lines = []
    
    # 2. ITERAZIONE SULLE RIGHE DEI DATI
    for row in rows[1:]:
        cols = row.find_all(["td", "th"])
        
        # Saltiamo le righe vuote o malformate
        if not cols:
            continue
            
        row_texts = [col.get_text(strip=True) for col in cols]
        
        # Se la riga contiene solo testo vuoto, la ignoriamo
        if not any(row_texts):
            continue
            
        # La prima colonna è quasi sempre il nome della metrica (es. "Total revenues")
        metric_name = row_texts[0] if row_texts[0] else "Sub-category / Note"
        
        # Inizializziamo la "frase" per questa specifica riga
        line_parts = [f"Metric: {metric_name}"]
        
        # Mappiamo i valori successivi ai rispettivi header
        for i in range(1, min(len(headers), len(row_texts))):
            val = row_texts[i]
            header_val = headers[i]
            
            # Filtriamo il rumore SEC: ignoriamo le colonne vuote, simboli solitari $, % o i trattini lunghi
            if val and val not in ['$', '%', '—', '-', '()'] and header_val != "BLANK_COL":
                line_parts.append(f"{header_val}: {val}")
        
        # Se la riga ha dei dati effettivi (non è solo un'intestazione di sezione interna alla tabella)
        if len(line_parts) > 1:
            # Uniamo i pezzi in una singola riga testuale
            linearized_lines.append(" | ".join(line_parts))
            
    return "\n".join(linearized_lines)

def clean_sec_text_v2(raw_html):
    """
    Parser SOTA per documenti SEC. Sostituisce la versione precedente.
    """
    soup = BeautifulSoup(raw_html, "lxml")
    
    # 1. Rimozione tag inutili
    for element in soup(["script", "style", "header", "footer"]):
        element.decompose()

    # 2. Intercettazione e Linearizzazione Tabelle
    for table in soup.find_all("table"):
        linearized_text = linearize_sec_table(table)
        if linearized_text:
            # Sostituiamo la tabella HTML con un blocco testuale delimitato.
            # Questi delimitatori [BEGIN/END TABLE] saranno d'oro per il nostro chunker!
            formatted_table = f"\n[BEGIN TABLE]\n{linearized_text}\n[END TABLE]\n"
            table.replace_with(formatted_table)
        else:
            table.decompose() 

    # 3. Estrazione testo finale
    text = soup.get_text(separator="\n")
    
    # 4. Pulizia del rumore SEC (XBRL, spazi multipli, IDs)
    text = re.sub(r'\b(us-gaap|srt|dei|iso4217):\S+', '', text)
    text = re.sub(r'\b\d{10}\b', '', text) # Rimuove codici CIK a 10 cifre
    
    # Riduciamo gli a capo multipli (lasciando max un doppio a capo per dividere i paragrafi)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()

In [2]:
# 1. Definiamo un frammento HTML "sporco" tipico della SEC
mock_sec_html = """
<html>
<body>
    <p>In 2024, we recognized total revenues of $96.77 billion.</p>
    <table>
        <tr>
            <td></td>
            <td></td>
            <td><strong>2024</strong></td>
            <td></td>
            <td><strong>2023</strong></td>
        </tr>
        <tr>
            <td><strong>Revenues</strong></td>
            <td></td><td></td><td></td><td></td>
        </tr>
        <tr>
            <td>Automotive revenues</td>
            <td>$</td>
            <td>82,419</td>
            <td>$</td>
            <td>78,509</td>
        </tr>
        <tr>
            <td>Energy generation and storage</td>
            <td></td>
            <td>6,035</td>
            <td></td>
            <td>3,909</td>
        </tr>
    </table>
    <p>Gross margin was impacted by vehicle pricing and mix.</p>
</body>
</html>
"""

# 2. Eseguiamo il nostro nuovo parser SOTA
print("🔍 RISULTATO DEL PARSER LINEARIZZATO:\n")
print(60 * "-")
test_output = clean_sec_text_v2(mock_sec_html)
print(test_output)
print(60 * "-")

🔍 RISULTATO DEL PARSER LINEARIZZATO:

------------------------------------------------------------
In 2024, we recognized total revenues of $96.77 billion.

[BEGIN TABLE]
Metric: Automotive revenues | 2024: 82,419 | 2023: 78,509
Metric: Energy generation and storage | 2024: 6,035 | 2023: 3,909
[END TABLE]

Gross margin was impacted by vehicle pricing and mix.
------------------------------------------------------------


In [3]:
import os
from sec_edgar_downloader import Downloader
from pathlib import Path

# 1. Setup del downloader (Usa una mail reale per rispettare le policy SEC)
# Formato richiesto dalla SEC: "NomeAzienda TuaMail@dominio.com"
user_agent = "RAG_Project test@example.com" 
download_dir = "./sec_data"

dl = Downloader("MyCompany", user_agent, download_dir)

# 2. Download dell'ultimo 10-K di TSLA (Anno fiscale 2024, depositato a inizio 2025)
print("📥 Scaricamento del 10-K di TSLA in corso...")
dl.get("10-K", "TSLA", limit=1)
print("✅ Download completato!")

# 3. Individuazione del file scaricato
base_path = Path(download_dir) / "sec-edgar-filings" / "TSLA" / "10-K"
try:
    submission_dir = next(base_path.iterdir())
    file_path = submission_dir / "full-submission.txt"
    print(f"📄 File trovato: {file_path}")
except StopIteration:
    print("❌ Errore: File non trovato.")

# 4. Lettura del file
with open(file_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

# 5. Esecuzione del nostro parser SOTA
print("🧹 Parsing del documento SEC in corso... (potrebbe richiedere qualche secondo)")
real_parsed_text = clean_sec_text_v2(raw_content)
print("✅ Parsing completato!")

# 6. Analisi dei risultati: cerchiamo le tabelle linearizzate!
# Dividiamo il testo usando i nostri marker
tables = [t for t in real_parsed_text.split("[BEGIN TABLE]") if "[END TABLE]" in t]

print(f"\n📊 Trovate {len(tables)} tabelle linearizzate nel documento.")

if tables:
    print("\n🔍 ESTRATTO DI UNA TABELLA FINANZIARIA (Linearizzata):\n")
    print(60 * "-")
    # Cerchiamo una tabella che contenga parole chiave finanziarie per assicurarci che non sia l'indice
    for t in tables:
        if "Revenues" in t or "Automotive" in t or "Assets" in t:
            # Estraiamo il contenuto della tabella e ne stampiamo le prime righe
            table_content = t.split("[END TABLE]")[0].strip()
            # Stampiamo solo le prime 15 righe per non inondare l'output
            lines = table_content.split('\n')[:15]
            print("[BEGIN TABLE]")
            print('\n'.join(lines))
            print("...\n[END TABLE]")
            break
    print(60 * "-")

📥 Scaricamento del 10-K di TSLA in corso...
✅ Download completato!
📄 File trovato: sec_data/sec-edgar-filings/TSLA/10-K/0001628280-26-003952/full-submission.txt
🧹 Parsing del documento SEC in corso... (potrebbe richiedere qualche secondo)
✅ Parsing completato!

📊 Trovate 97 tabelle linearizzate nel documento.

🔍 ESTRATTO DI UNA TABELLA FINANZIARIA (Linearizzata):

------------------------------------------------------------
[BEGIN TABLE]
Metric: Dec. 31, 2025 | 12 Months Ended: Dec. 31, 2024
Metric: Revenues | 12 Months Ended: $ 94,827
Metric: Total cost of revenues | 12 Months Ended: 77,733
Metric: Gross profit | 12 Months Ended: 17,094
Metric: Research and development | 12 Months Ended: 6,411
Metric: Selling, general and administrative | 12 Months Ended: 5,834
Metric: Restructuring and other | 12 Months Ended: 494
Metric: Total operating expenses | 12 Months Ended: 12,739
Metric: Income from operations | 12 Months Ended: 4,355
Metric: Interest income | 12 Months Ended: 1,680
Metric: 

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_smart_chunks_v2(text, ticker, report_type, fiscal_year="2024"):
    """
    Divide il testo rispettando le tabelle e inietta il contesto per potenziare l'embedding.
    """
    # 1. Prepariamo l'iniezione del contesto (Questo cambia tutto per il Vector Search)
    context_prefix = f"[COMPANY: {ticker} | FISCAL YEAR: {fiscal_year} | SEC FORM: {report_type}]\n"
    
    # 2. Configuriamo lo splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, # Riduciamo un po' per avere embedding più densi e precisi
        chunk_overlap=200,
        # Aggiungiamo i nostri delimitatori in cima alla priorità di taglio!
        separators=["\n\n", "\n[BEGIN TABLE]\n", "\n[END TABLE]\n", "\n", ". ", " "],
        is_separator_regex=False,
    )
    
    raw_chunks = text_splitter.split_text(text)
    
    final_chunks = []
    for i, chunk_text in enumerate(raw_chunks):
        # 3. Context Injection: Saldiamo il contesto al testo
        enriched_content = context_prefix + chunk_text.strip()
        
        final_chunks.append({
            "content": enriched_content,
            "metadata": {
                "ticker": ticker,
                "report_type": report_type,
                "year": fiscal_year, 
                "chunk_index": i
            }
        })
        
    return final_chunks

# TESTIAMOLO SUL TUO ESTRATTO
test_table = """[BEGIN TABLE]
Metric: Revenues | 12 Months Ended: $ 94,827
Metric: Total cost of revenues | 12 Months Ended: 77,733
Metric: Gross profit | 12 Months Ended: 17,094
[END TABLE]"""

chunks = create_smart_chunks_v2(test_table, "TSLA", "10-K")
print("Esempio di Chunk Inietatto:\n")
print(chunks[0]['content'])

Esempio di Chunk Inietatto:

[COMPANY: TSLA | FISCAL YEAR: 2024 | SEC FORM: 10-K]
[BEGIN TABLE]
Metric: Revenues | 12 Months Ended: $ 94,827
Metric: Total cost of revenues | 12 Months Ended: 77,733
Metric: Gross profit | 12 Months Ended: 17,094
[END TABLE]


In [5]:
import re
import json
from pathlib import Path
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# 1. MODULO PARSER (Linearizzazione + Sezioni)
# ==========================================
def linearize_sec_table(table_soup):
    rows = table_soup.find_all("tr")
    if not rows: return ""

    headers = []
    if rows:
        for col in rows[0].find_all(["td", "th"]):
            text = col.get_text(strip=True)
            headers.append(text if text else "BLANK_COL")

    linearized_lines = []
    for row in rows[1:]:
        cols = row.find_all(["td", "th"])
        if not cols: continue
            
        row_texts = [col.get_text(strip=True) for col in cols]
        if not any(row_texts): continue
            
        metric_name = row_texts[0] if row_texts[0] else "Sub-category / Note"
        line_parts = [f"Metric: {metric_name}"]
        
        for i in range(1, min(len(headers), len(row_texts))):
            val = row_texts[i]
            header_val = headers[i]
            if val and val not in ['$', '%', '—', '-', '()'] and header_val != "BLANK_COL":
                line_parts.append(f"{header_val}: {val}")
        
        if len(line_parts) > 1:
            linearized_lines.append(" | ".join(line_parts))
            
    return "\n".join(linearized_lines)

def clean_sec_text_final(raw_html):
    soup = BeautifulSoup(raw_html, "lxml")
    
    for element in soup(["script", "style", "header", "footer"]):
        element.decompose()

    for table in soup.find_all("table"):
        linearized_text = linearize_sec_table(table)
        if linearized_text:
            formatted_table = f"\n[BEGIN TABLE]\n{linearized_text}\n[END TABLE]\n"
            table.replace_with(formatted_table)
        else:
            table.decompose() 

    text = soup.get_text(separator="\n")
    
    text = re.sub(r'\b(us-gaap|srt|dei|iso4217):\S+', '', text)
    text = re.sub(r'\b\d{10}\b', '', text) 
    
    # Marcatura delle Sezioni SEC
    text = re.sub(r'\n\s*(Item\s+[1-9][A-Z]?\.\s+[^\n]{5,80})\n', r'\n[SECTION: \1]\n', text, flags=re.IGNORECASE)
    
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# ==========================================
# 2. MODULO CHUNKER (State-Aware + Context Injection)
# ==========================================
def create_smart_chunks_final(text, ticker, report_type, fiscal_year="2024"):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, 
        chunk_overlap=200,
        separators=["\n\n", "\n[SECTION:", "\n[BEGIN TABLE]\n", "\n[END TABLE]\n", "\n", ". ", " "],
        is_separator_regex=False,
    )
    
    raw_chunks = text_splitter.split_text(text)
    
    final_chunks = []
    current_section = "General Overview" 
    section_pattern = re.compile(r'\[SECTION:\s*(.*?)\]')

    for i, chunk_text in enumerate(raw_chunks):
        section_match = section_pattern.search(chunk_text)
        if section_match:
            current_section = section_match.group(1).strip()
            chunk_text = section_pattern.sub('', chunk_text).strip()
            
        context_prefix = f"[COMPANY: {ticker} | FY: {fiscal_year} | FORM: {report_type} | SECTION: {current_section}]\n"
        enriched_content = context_prefix + chunk_text.strip()
        
        final_chunks.append({
            "content": enriched_content,
            "metadata": {
                "ticker": ticker,
                "report_type": report_type,
                "year": fiscal_year, 
                "section": current_section, 
                "chunk_index": i
            }
        })
        
    return final_chunks

# ==========================================
# 3. ESECUZIONE SUL FILE REALE
# ==========================================
print("🚀 Avvio pipeline end-to-end sul Notebook...")

# Assumiamo che il file esista già dal download precedente
file_path = Path("./sec_data/sec-edgar-filings/TSLA/10-K") 
try:
    submission_dir = next(file_path.iterdir())
    full_path = submission_dir / "full-submission.txt"
    with open(full_path, "r", encoding="utf-8") as f:
        raw_content = f.read()
except Exception as e:
    print(f"❌ Errore nel trovare il file. Assicurati che il download della cella precedente sia andato a buon fine. ({e})")
    raw_content = None

if raw_content:
    print("🧹 1/3 Parsing del testo (Linearizzazione & Sezioni)...")
    refined_text = clean_sec_text_final(raw_content)

    print("✂️ 2/3 Chunking intelligente...")
    chunks = create_smart_chunks_final(refined_text, "TSLA", "10-K", "2024")

    print(f"✅ Creati {len(chunks)} chunks!")
    
    # Costruiamo l'oggetto finale
    final_json_data = {
        "ticker": "TSLA",
        "report_type": "10-K",
        "fiscal_year": "2024",
        "total_chunks": len(chunks),
        "chunks": chunks
    }

    # Salviamo un file di test localmente per ispezione
    output_test_file = "test_tsla_chunks.json"
    with open(output_test_file, "w", encoding="utf-8") as f:
        json.dump(final_json_data, f, ensure_ascii=False, indent=4)
        
    print(f"💾 File JSON salvato localmente come: {output_test_file}")

    print("\n🔍 3/3 ISPEZIONE DEL RISULTATO (Primo Chunk e un Chunk a caso con Tabella):")
    print(60*"=")
    
    # Stampiamo il primo chunk
    print("📌 ESEMPIO CHUNK TESTUALE (Inizio Documento):")
    print(json.dumps(chunks[0], indent=2))
    print(60*"-")
    
    # Cerchiamo e stampiamo il primo chunk che contiene una tabella linearizzata
    for c in chunks:
        if "[BEGIN TABLE]" in c["content"]:
            print("📌 ESEMPIO CHUNK CON TABELLA LINEARIZZATA:")
            print(json.dumps(c, indent=2))
            break

🚀 Avvio pipeline end-to-end sul Notebook...
🧹 1/3 Parsing del testo (Linearizzazione & Sezioni)...
✂️ 2/3 Chunking intelligente...
✅ Creati 4361 chunks!
💾 File JSON salvato localmente come: test_tsla_chunks.json

🔍 3/3 ISPEZIONE DEL RISULTATO (Primo Chunk e un Chunk a caso con Tabella):
📌 ESEMPIO CHUNK TESTUALE (Inizio Documento):
{
  "content": "[COMPANY: TSLA | FY: 2024 | FORM: 10-K | SECTION: General Overview]\n-26-003952.txt : 20260129\n\n-26-003952.hdr.sgml : 20260129\n\n20260128205503\nACCESSION NUMBER:\t\t-26-003952\nCONFORMED SUBMISSION TYPE:\t10-K\nPUBLIC DOCUMENT COUNT:\t\t119\nCONFORMED PERIOD OF REPORT:\t20251231\nFILED AS OF DATE:\t\t20260129\nDATE AS OF CHANGE:\t\t20260128\n\nFILER:\n\n\tCOMPANY DATA:\t\n\t\tCOMPANY CONFORMED NAME:\t\t\tTesla, Inc.\n\t\tCENTRAL INDEX KEY:\t\t\t\n\t\tSTANDARD INDUSTRIAL CLASSIFICATION:\tMOTOR VEHICLES & PASSENGER CAR BODIES [3711]\n\t\tORGANIZATION NAME:           \t04 Manufacturing\n\t\tEIN:\t\t\t\t912197729\n\t\tSTATE OF INCORPORATION:\t

In [ ]:
import re
import json
from pathlib import Path
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# 1. MODULO PARSER (Filtro 10-K, Linearizzazione + Sezioni + Scudo Anti-HTML)
# ==========================================
def extract_pure_10k(raw_submission):
    """
    Estrae SOLO il vero documento 10-K dal contenitore full-submission della SEC,
    ignorando XML, immagini base64 e altri allegati.
    """
    match = re.search(r'<DOCUMENT>\s*<TYPE>10-K.*?(<TEXT>.*?</TEXT>)', raw_submission, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1)
    return raw_submission

def linearize_sec_table(table_soup):
    rows = table_soup.find_all("tr")
    if not rows: return ""

    headers = []
    if rows:
        for col in rows[0].find_all(["td", "th"]):
            text = col.get_text(strip=True)
            headers.append(text if text else "BLANK_COL")

    linearized_lines = []
    for row in rows[1:]:
        cols = row.find_all(["td", "th"])
        if not cols: continue
            
        row_texts = [col.get_text(strip=True) for col in cols]
        if not any(row_texts): continue
            
        metric_name = row_texts[0] if row_texts[0] else "Sub-category / Note"
        line_parts = [f"Metric: {metric_name}"]
        
        for i in range(1, min(len(headers), len(row_texts))):
            val = row_texts[i]
            header_val = headers[i]
            if val and val not in ['$', '%', '—', '-', '()'] and header_val != "BLANK_COL":
                line_parts.append(f"{header_val}: {val}")
        
        if len(line_parts) > 1:
            linearized_lines.append(" | ".join(line_parts))
            
    return "\n".join(linearized_lines)

def clean_sec_text_final(raw_html):
    soup = BeautifulSoup(raw_html, "lxml")
    
    # 1. Rimuoviamo tag inutili
    for element in soup(["script", "style", "header", "footer", "ix:header"]):
        element.decompose()

    # 2. Linearizzazione Tabelle
    for table in soup.find_all("table"):
        linearized_text = linearize_sec_table(table)
        if linearized_text:
            formatted_table = f"\n[BEGIN TABLE]\n{linearized_text}\n[END TABLE]\n"
            table.replace_with(formatted_table)
        else:
            table.decompose() 

    text = soup.get_text(separator="\n")
    
    # 3. LO SCUDO ANTI-HTML BLEEDING (Novità)
    # Rimuove rimasugli di attributi HTML (es. style="...", class="...")
    text = re.sub(r'[a-zA-Z\-]+="[^"]*"', '', text)
    # Rimuove tag orfani (es. </td>, </span>)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 4. Pulizia XBRL e rumore SEC
    text = re.sub(r'\b(us-gaap|srt|dei|iso4217):\S+', '', text)
    text = re.sub(r'\b\d{10}\b', '', text) 
    
    # 5. Marcatura delle Sezioni SEC
    text = re.sub(r'\n\s*(Item\s+[1-9][A-Z]?\.\s+[^\n]{5,80})\n', r'\n[SECTION: \1]\n', text, flags=re.IGNORECASE)
    
    # 6. Compressione spazi e a capo
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    
    return text.strip()

# ==========================================
# 2. MODULO CHUNKER (State-Aware + Context Injection)
# ==========================================
def create_smart_chunks_final(text, ticker, report_type, fiscal_year="2025"):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, 
        chunk_overlap=200,
        separators=["\n\n", "\n[SECTION:", "\n[BEGIN TABLE]\n", "\n[END TABLE]\n", "\n", ". ", " "],
        is_separator_regex=False,
    )
    
    raw_chunks = text_splitter.split_text(text)
    
    final_chunks = []
    current_section = "General Overview" 
    section_pattern = re.compile(r'\[SECTION:\s*(.*?)\]')

    for i, chunk_text in enumerate(raw_chunks):
        section_match = section_pattern.search(chunk_text)
        if section_match:
            current_section = section_match.group(1).strip()
            chunk_text = section_pattern.sub('', chunk_text).strip()
            
        context_prefix = f"[COMPANY: {ticker} | FY: {fiscal_year} | FORM: {report_type} | SECTION: {current_section}]\n"
        enriched_content = context_prefix + chunk_text.strip()
        
        final_chunks.append({
            "content": enriched_content,
            "metadata": {
                "ticker": ticker,
                "report_type": report_type,
                "year": fiscal_year, 
                "section": current_section, 
                "chunk_index": i
            }
        })
        
    return final_chunks

# ==========================================
# 3. ESECUZIONE SUL FILE REALE (Parametrica)
# ==========================================
from sec_edgar_downloader import Downloader
import shutil

print("🚀 Avvio pipeline end-to-end sul Notebook...")

# --- PANNELLO DI CONTROLLO ---
TARGET_TICKER = "TSLA"
TARGET_YEAR = "2023" # <--- CAMBIA QUESTO VALORE PER CAMBIARE ANNO FISCALE
REPORT_TYPE = "10-K"
# -----------------------------

# Calcoliamo la data di deposito: il 10-K dell'anno X viene depositato a inizio anno X+1.
# Diciamo al downloader di prendere l'ultimo 10-K disponibile PRIMA di Aprile dell'anno successivo.
filing_deadline = f"{int(TARGET_YEAR) + 1}-04-01"

print(f"📥 1/5 Download del {REPORT_TYPE} per l'anno fiscale {TARGET_YEAR}...")
# Inserisci qui la tua mail come prima
dl = Downloader("MyCompany", "test@example.com", "./sec_data") 
dl.get(REPORT_TYPE, TARGET_TICKER, limit=1, before=filing_deadline)

# Troviamo dinamicamente la cartella appena scaricata
file_path = Path(f"./sec_data/sec-edgar-filings/{TARGET_TICKER}/{REPORT_TYPE}") 

# Poiché stiamo scaricando documenti specifici, prendiamo l'ultima cartella modificata/creata
try:
    # Ordina le cartelle di submission in modo da prendere la più recente appena scaricata
    submission_dirs = sorted([d for d in file_path.iterdir() if d.is_dir()], key=os.path.getmtime, reverse=True)
    full_path = submission_dirs[0] / "full-submission.txt"
    
    with open(full_path, "r", encoding="utf-8") as f:
        raw_content = f.read()
except Exception as e:
    print(f"❌ Errore nel trovare il file: {e}")
    raw_content = None

if raw_content:
    print(f"🛡️ 2/5 Estrazione del documento 10-K puro ({TARGET_YEAR})...")
    pure_10k_html = extract_pure_10k(raw_content)

    print("🧹 3/5 Parsing del testo (Linearizzazione, Sezioni & Scudo HTML)...")
    refined_text = clean_sec_text_final(pure_10k_html)

    print("✂️ 4/5 Chunking intelligente...")
    # Passiamo le nostre variabili dinamicamente alla funzione!
    chunks = create_smart_chunks_final(refined_text, TARGET_TICKER, REPORT_TYPE, TARGET_YEAR)

    print(f"✅ Creati {len(chunks)} chunks!")
    
    # Costruiamo il JSON finale usando le variabili
    final_json_data = {
        "ticker": TARGET_TICKER,
        "report_type": REPORT_TYPE,
        "fiscal_year": TARGET_YEAR,
        "total_chunks": len(chunks),
        "chunks": chunks
    }

    # Salviamo il file col nome dell'anno per non sovrascrivere i vecchi test
    output_test_file = f"test_{TARGET_TICKER.lower()}_{TARGET_YEAR}_chunks.json"
    with open(output_test_file, "w", encoding="utf-8") as f:
        json.dump(final_json_data, f, ensure_ascii=False, indent=4)
        
    print(f"💾 File JSON salvato localmente come: {output_test_file}")

    print(f"\n🔍 5/5 ISPEZIONE DEL RISULTATO ({TARGET_YEAR}):")
    print(60*"=")
    
    print("📌 ESEMPIO CHUNK CASUALE:")
    print(json.dumps(chunks[len(chunks)//2], indent=2))
        
    print(60*"-")

🚀 Avvio pipeline end-to-end sul Notebook...
🛡️ 1/4 Estrazione del documento 10-K puro...
🧹 2/4 Parsing del testo (Linearizzazione, Sezioni & Scudo HTML)...
✂️ 3/4 Chunking intelligente...
✅ Creati 323 chunks!
💾 File JSON salvato localmente come: tsla_10-k_chunks.json

🔍 4/4 ISPEZIONE DEL RISULTATO:
📌 ESEMPIO CHUNK RIPARATO (Precedentemente affetto da HTML Bleeding):
{
  "content": "[COMPANY: TSLA | FY: 2024 | FORM: 10-K | SECTION: ITEM 6. [RESERVED]\nStock-Based Compensation\nWe use the fair value method of accounting for our restricted stock awards (\u201cRSAs\u201d), stock options and restricted stock units (\u201cRSUs\u201d) granted to employees and for our employee stock purchase plan (\u201cthe ESPP\u201d) to measure the cost of employee services received in exchange for the stock-based awards. Stock-based compensation expense for equity awards is a non-cash expense and is recorded in Cost of revenues, Research and development expense and Selling, general and administrative expens